# NoiseEnhancer U-Net Training (Kaggle GPU)

Clones [decode2211/NoiseEnhancer](https://github.com/decode2211/NoiseEnhancer), trains the U-Net mask predictor on a GPU, plots the training curve, and evaluates the best checkpoint on the test set — the same run this project's local CPU fallback does, just fast.

**Before running:** attach the `jiangwq666/voicebank-demand` dataset to this kernel (or `kernel-metadata.json` already declares it if pushed via `kaggle kernels push -p kaggle/`), and turn on a GPU accelerator (T4 x2 or P100) plus internet access in the kernel's settings.

Everything under `/kaggle/working/NoiseEnhancer/` (checkpoint, `results/train_log.csv`, `results/training_curve.png`, `results/model_metrics.csv`) is what Kaggle persists as this kernel's output — download it with `kaggle kernels output devvashist/noiseenhancer-train`.

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/decode2211/NoiseEnhancer.git"
REPO_DIR = Path("/kaggle/working/NoiseEnhancer")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"cwd: {os.getcwd()}")

In [ ]:
# torch/torchaudio ship preinstalled with CUDA on Kaggle images — do not
# reinstall them, only add what's missing from requirements.txt.
!pip install -q soundfile pesq pystoi pyyaml

In [ ]:
INPUT_ROOT = Path("/kaggle/input")


def find_wav_dir(root, must_have):
    """Same trick as 01_dataset_audit.ipynb's resolve_wav_dir: don't assume
    exact folder names (e.g. '_28spk_wav') since the Kaggle mirror of this
    dataset may not name them the same way as the official download."""
    for p in sorted(root.rglob("*")):
        if not p.is_dir():
            continue
        name = p.name.lower()
        if all(k in name for k in must_have) and any(p.glob("*.wav")):
            return p
    raise FileNotFoundError(f"No directory under {root} matching {must_have} with .wav files")


clean_train_dir = find_wav_dir(INPUT_ROOT, ("clean", "train"))
noisy_train_dir = find_wav_dir(INPUT_ROOT, ("noisy", "train"))
clean_test_dir = find_wav_dir(INPUT_ROOT, ("clean", "test"))
noisy_test_dir = find_wav_dir(INPUT_ROOT, ("noisy", "test"))

print(f"clean_train_dir = {clean_train_dir}")
print(f"noisy_train_dir = {noisy_train_dir}")
print(f"clean_test_dir  = {clean_test_dir}")
print(f"noisy_test_dir  = {noisy_test_dir}")

In [ ]:
CHECKPOINT_PATH = "/kaggle/working/NoiseEnhancer/checkpoints/unet_se.pt"

config_text = f"""\
data:
  noisy_train_dir: {noisy_train_dir}
  clean_train_dir: {clean_train_dir}
  noisy_test_dir: {noisy_test_dir}
  clean_test_dir: {clean_test_dir}

train:
  segment_seconds: 2.0
  batch_size: 8
  epochs: 30
  lr: 1.0e-3
  si_sdr_weight: 0.0
  base_ch: 32
  checkpoint_path: {CHECKPOINT_PATH}
  log_path: results/train_log.csv
  num_workers: 4
"""
Path("configs/config.yaml").write_text(config_text)
print(config_text)

In [ ]:
import torch

print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")

In [ ]:
import time

t0 = time.time()
!python -m src.train
train_wall_sec = time.time() - t0
print(f"\n[kaggle] training wall-clock: {train_wall_sec:.1f}s ({train_wall_sec/3600:.2f}h)")

In [ ]:
import csv

with open("results/train_log.csv") as f:
    rows = list(csv.DictReader(f))

print(f"epochs completed: {len(rows)}")
best_row = min(rows, key=lambda r: float(r["val_loss"]))
print(f"best val_loss: {best_row['val_loss']} at epoch {best_row['epoch']}")
if len(rows) >= 5:
    print(f"last 5 train_loss: {[r['train_loss'] for r in rows[-5:]]}")
    print(f"last 5 val_loss:   {[r['val_loss'] for r in rows[-5:]]}")

In [ ]:
!python -m src.plot_training_curve results/train_log.csv --output results/training_curve.png

from IPython.display import Image, display
display(Image("results/training_curve.png"))

In [ ]:
t0 = time.time()
!python -m src.evaluate --mode model --checkpoint {CHECKPOINT_PATH}
eval_wall_sec = time.time() - t0
print(f"[kaggle] evaluation wall-clock: {eval_wall_sec:.1f}s")

## Done

Pull the results back into the local repo: `kaggle kernels output devvashist/noiseenhancer-train -p results/` (or download from the kernel's Output tab), then commit `results/train_log.csv`, `results/training_curve.png`, and `results/model_metrics.csv` — the `.pt` checkpoint itself stays gitignored.